# 📄 Medical Report Image Enhancement: Deblurring & Denoising
## Optimized for Google Colab Free Tier

### Your Dataset: Medical_Report/ on Google Drive
- train/input/ & train/target/ (336 paired images)
- val/input/ & val/target/ (90 paired images)

### Quick Start: Runtime → T4 GPU → Run All
### Expected Time: ~30-45 min total

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted')
except ImportError:
    print('⚠️  google.colab not available — not running in Colab, skipping Drive mount')
except Exception as e:
    print(f'⚠️  Drive mount failed ({type(e).__name__}: {e})')
    print('    Check your Google account permissions and try again.')

# GPU detection
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU detected: {gpus[0].name}')
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                             '--format=csv,noheader'], capture_output=True, text=True)
    print(result.stdout.strip())
    # Enable mixed precision
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('✅ Mixed precision enabled (halves VRAM usage)')
else:
    print('⚠️  No GPU detected! Training will be slow.')
    print('    Go to Runtime → Change runtime type → T4 GPU')

import numpy as np
import cv2
import os
import gc
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Check RAM
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f'✅ System RAM: {ram_gb:.1f} GB')
except ImportError:
    print('psutil not available — cannot report RAM')

In [ ]:
# ===== ALL SETTINGS IN ONE PLACE =====
DATASET_PATH     = '/content/drive/MyDrive/Medical_Report/'
TRAIN_INPUT_DIR  = os.path.join(DATASET_PATH, 'train', 'input')
TRAIN_TARGET_DIR = os.path.join(DATASET_PATH, 'train', 'target')
VAL_INPUT_DIR    = os.path.join(DATASET_PATH, 'val',   'input')
VAL_TARGET_DIR   = os.path.join(DATASET_PATH, 'val',   'target')

CHECKPOINT_DIR = '/content/drive/MyDrive/Medical_Report_checkpoints/'
RESULTS_DIR    = '/content/drive/MyDrive/Medical_Report_results/'

RESIZE_TO      = 512   # Resize images to 512×512 before patching
PATCH_SIZE     = 128   # Patch size for training
PATCH_STRIDE   = 128   # No overlap — keeps patch count manageable (~5-6 K)
BATCH_SIZE     = 32    # Larger batch = fewer steps = faster epochs
EPOCHS         = 150
LEARNING_RATE  = 1e-3
IMG_CHANNELS   = 1     # Grayscale

# Create output directories
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)
print('✅ Configuration set')

In [ ]:
# Verify folders exist and count files
SUPPORTED_EXTS = {'.jpg', '.jpeg', '.png', '.tiff', '.tif', '.bmp'}

def list_images(folder):
    """Return sorted list of image filenames in *folder*."""
    if not os.path.isdir(folder):
        raise FileNotFoundError(
            f'Folder not found: {folder}\n'
            f'Fix: check DATASET_PATH in Cell 2 and ensure the folder exists on Drive.'
        )
    files = [f for f in sorted(os.listdir(folder))
             if os.path.splitext(f)[1].lower() in SUPPORTED_EXTS]
    if not files:
        raise ValueError(
            f'No supported images found in {folder}\n'
            f'Supported formats: {SUPPORTED_EXTS}'
        )
    return files

try:
    train_inp_files = list_images(TRAIN_INPUT_DIR)
    train_tgt_files = list_images(TRAIN_TARGET_DIR)
    val_inp_files   = list_images(VAL_INPUT_DIR)
    val_tgt_files   = list_images(VAL_TARGET_DIR)

    # Match filenames between input/ and target/
    train_files = sorted(set(train_inp_files) & set(train_tgt_files))
    val_files   = sorted(set(val_inp_files)   & set(val_tgt_files))

    if len(train_files) == 0:
        raise ValueError('No matching filename pairs found in train/input and train/target!')
    if len(val_files) == 0:
        raise ValueError('No matching filename pairs found in val/input and val/target!')

    print(f'✅ Training pairs:   {len(train_files)}')
    print(f'✅ Validation pairs: {len(val_files)}')

    # Display 6 random sample pairs (input on top, target on bottom)
    import random
    sample_files = random.sample(train_files, min(6, len(train_files)))
    fig, axes = plt.subplots(2, len(sample_files), figsize=(3 * len(sample_files), 6))
    for col, fname in enumerate(sample_files):
        inp = cv2.imread(os.path.join(TRAIN_INPUT_DIR, fname), cv2.IMREAD_GRAYSCALE)
        tgt = cv2.imread(os.path.join(TRAIN_TARGET_DIR, fname), cv2.IMREAD_GRAYSCALE)
        axes[0, col].imshow(inp, cmap='gray')
        axes[0, col].set_title('Input', fontsize=8)
        axes[0, col].axis('off')
        axes[1, col].imshow(tgt, cmap='gray')
        axes[1, col].set_title('Target', fontsize=8)
        axes[1, col].axis('off')
    plt.suptitle('Sample Training Pairs', fontsize=12)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'❌ Dataset error: {e}')
    raise

In [ ]:
def extract_patches_to_ram(input_dir, target_dir, file_list, resize_to, patch_size, patch_stride):
    """
    Reads images from Drive, resizes to resize_to×resize_to,
    extracts patches, returns NumPy arrays.
    """
    input_patches  = []
    target_patches = []

    for i, fname in enumerate(file_list):
        if i % 50 == 0:
            print(f'  Processing {i + 1}/{len(file_list)}...')

        inp_img = cv2.imread(os.path.join(input_dir,  fname), cv2.IMREAD_GRAYSCALE)
        tgt_img = cv2.imread(os.path.join(target_dir, fname), cv2.IMREAD_GRAYSCALE)

        if inp_img is None or tgt_img is None:
            print(f'  ⚠️  Skipping {fname} (could not read)')
            continue

        # Resize to fixed size
        inp_img = cv2.resize(inp_img, (resize_to, resize_to), interpolation=cv2.INTER_AREA)
        tgt_img = cv2.resize(tgt_img, (resize_to, resize_to), interpolation=cv2.INTER_AREA)

        # Normalize to [0, 1]
        inp_img = inp_img.astype(np.float32) / 255.0
        tgt_img = tgt_img.astype(np.float32) / 255.0

        # Extract patches
        for y in range(0, resize_to - patch_size + 1, patch_stride):
            for x in range(0, resize_to - patch_size + 1, patch_stride):
                input_patches.append(inp_img[y:y + patch_size, x:x + patch_size])
                target_patches.append(tgt_img[y:y + patch_size, x:x + patch_size])

    # Convert to arrays and add channel dimension
    input_patches  = np.array(input_patches,  dtype=np.float32)[..., np.newaxis]  # (N, 128, 128, 1)
    target_patches = np.array(target_patches, dtype=np.float32)[..., np.newaxis]  # (N, 128, 128, 1)

    return input_patches, target_patches

try:
    print('📦 Extracting training patches into RAM...')
    train_input_patches, train_target_patches = extract_patches_to_ram(
        TRAIN_INPUT_DIR, TRAIN_TARGET_DIR, train_files,
        RESIZE_TO, PATCH_SIZE, PATCH_STRIDE)

    print('📦 Extracting validation patches into RAM...')
    val_input_patches, val_target_patches = extract_patches_to_ram(
        VAL_INPUT_DIR, VAL_TARGET_DIR, val_files,
        RESIZE_TO, PATCH_SIZE, PATCH_STRIDE)

    total_mb = (train_input_patches.nbytes + train_target_patches.nbytes +
                val_input_patches.nbytes   + val_target_patches.nbytes) / 1024**2

    print(f'✅ Training patches:   {len(train_input_patches)}')
    print(f'✅ Validation patches: {len(val_input_patches)}')
    print(f'✅ Memory used: {total_mb:.1f} MB')
    # Expected: ~5,376 train / ~1,440 val / ~180 MB

except Exception as e:
    print(f'❌ Patch extraction error: {e}')
    raise

In [ ]:
# Data augmentation function
@tf.function
def augment(input_patch, target_patch):
    """Apply identical random flips, rotations, and brightness to both patches."""
    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        input_patch  = tf.image.flip_left_right(input_patch)
        target_patch = tf.image.flip_left_right(target_patch)
    # Random vertical flip
    if tf.random.uniform(()) > 0.5:
        input_patch  = tf.image.flip_up_down(input_patch)
        target_patch = tf.image.flip_up_down(target_patch)
    # Random 90° rotation
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    input_patch  = tf.image.rot90(input_patch,  k)
    target_patch = tf.image.rot90(target_patch, k)
    # Random brightness (subtle ±5%)
    delta = tf.random.uniform((), -0.05, 0.05)
    input_patch  = tf.clip_by_value(input_patch  + delta, 0.0, 1.0)
    target_patch = tf.clip_by_value(target_patch + delta, 0.0, 1.0)
    return input_patch, target_patch

try:
    # Build tf.data pipeline
    train_dataset = tf.data.Dataset.from_tensor_slices(
        (train_input_patches, train_target_patches))
    train_dataset = train_dataset.shuffle(buffer_size=5000, seed=42)
    train_dataset = train_dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.batch(BATCH_SIZE)
    train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices(
        (val_input_patches, val_target_patches))
    val_dataset = val_dataset.batch(BATCH_SIZE)
    val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

    n_train_batches = len(train_input_patches) // BATCH_SIZE
    n_val_batches   = len(val_input_patches)   // BATCH_SIZE
    print(f'✅ Train: {len(train_input_patches)} patches → {n_train_batches} batches/epoch')
    print(f'✅ Val:   {len(val_input_patches)} patches → {n_val_batches} batches/epoch')
    print(f'   Expected: ~{n_train_batches * 10 // 60}-{n_train_batches * 15 // 60} min/epoch at ~10-15s per epoch')

except Exception as e:
    print(f'❌ tf.data pipeline error: {e}')
    raise

In [ ]:
# Classical enhancement implementations
def clahe_enhance(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(img)

def nlm_enhance(img):
    return cv2.fastNlMeansDenoising(img, None, h=10, templateWindowSize=7, searchWindowSize=21)

def unsharp_enhance(img):
    blurred = cv2.GaussianBlur(img, (5, 5), 1.0)
    return cv2.addWeighted(img, 1.5, blurred, -0.5, 0)

def combined_enhance(img):
    """CLAHE → NLM denoising → unsharp masking pipeline."""
    out = clahe_enhance(img)
    out = nlm_enhance(out)
    out = unsharp_enhance(out)
    return out

def compute_psnr(img1, img2):
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    mse  = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(255.0 / np.sqrt(mse))

def compute_ssim(img1, img2):
    return ssim_metric(img1, img2, data_range=255)

try:
    methods = {
        'CLAHE':     clahe_enhance,
        'NLM':       nlm_enhance,
        'Unsharp':   unsharp_enhance,
        'Combined':  combined_enhance,
    }
    classical_results = {name: {'psnr': [], 'ssim': []} for name in methods}

    for fname in val_files[:20]:  # Use first 20 val images for speed
        inp = cv2.imread(os.path.join(VAL_INPUT_DIR,  fname), cv2.IMREAD_GRAYSCALE)
        tgt = cv2.imread(os.path.join(VAL_TARGET_DIR, fname), cv2.IMREAD_GRAYSCALE)
        if inp is None or tgt is None:
            continue
        tgt_resized = cv2.resize(tgt, (inp.shape[1], inp.shape[0]))
        for name, fn in methods.items():
            enhanced = fn(inp)
            enhanced_r = cv2.resize(enhanced, (tgt_resized.shape[1], tgt_resized.shape[0]))
            classical_results[name]['psnr'].append(compute_psnr(enhanced_r, tgt_resized))
            classical_results[name]['ssim'].append(compute_ssim(enhanced_r, tgt_resized))

    print('Classical Enhancement Results (avg over 20 val images):')
    print(f'  {"Method":<12} {"PSNR":>8} {"SSIM":>8}')
    for name in methods:
        avg_psnr = np.mean(classical_results[name]['psnr'])
        avg_ssim = np.mean(classical_results[name]['ssim'])
        print(f'  {name:<12} {avg_psnr:>8.2f} {avg_ssim:>8.4f}')

    # Show before/after for 3 sample images
    n_show = min(3, len(val_files))
    fig, axes = plt.subplots(n_show, len(methods) + 2,
                             figsize=(3 * (len(methods) + 2), 3 * n_show))
    for row, fname in enumerate(val_files[:n_show]):
        inp = cv2.imread(os.path.join(VAL_INPUT_DIR,  fname), cv2.IMREAD_GRAYSCALE)
        tgt = cv2.imread(os.path.join(VAL_TARGET_DIR, fname), cv2.IMREAD_GRAYSCALE)
        axes[row, 0].imshow(inp, cmap='gray'); axes[row, 0].set_title('Input');  axes[row, 0].axis('off')
        axes[row, 1].imshow(tgt, cmap='gray'); axes[row, 1].set_title('Target'); axes[row, 1].axis('off')
        for col, (name, fn) in enumerate(methods.items(), start=2):
            axes[row, col].imshow(fn(inp), cmap='gray')
            axes[row, col].set_title(name, fontsize=8)
            axes[row, col].axis('off')
    plt.suptitle('Classical Enhancement Methods', fontsize=13)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'❌ Classical enhancement error: {e}')
    raise

In [ ]:
from tensorflow.keras import layers, Model, regularizers

def build_unet(input_shape=(128, 128, 1)):
    inputs = layers.Input(shape=input_shape)

    # ── Encoder Level 1 ───────────────────────────────────────────────────
    c1 = layers.Conv2D(32, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation('relu')(c1)
    c1 = layers.Conv2D(32, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(c1)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Activation('relu')(c1)
    p1 = layers.MaxPooling2D(2)(c1)

    # ── Encoder Level 2 ───────────────────────────────────────────────────
    c2 = layers.Conv2D(64, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation('relu')(c2)
    c2 = layers.Conv2D(64, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(c2)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Activation('relu')(c2)
    p2 = layers.MaxPooling2D(2)(c2)

    # ── Encoder Level 3 ───────────────────────────────────────────────────
    c3 = layers.Conv2D(128, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(p2)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('relu')(c3)
    c3 = layers.Conv2D(128, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(c3)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Activation('relu')(c3)
    p3 = layers.MaxPooling2D(2)(c3)

    # ── Bottleneck ────────────────────────────────────────────────────────
    bn = layers.Conv2D(256, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(p3)
    bn = layers.BatchNormalization()(bn)
    bn = layers.Activation('relu')(bn)
    bn = layers.Conv2D(256, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(bn)
    bn = layers.BatchNormalization()(bn)
    bn = layers.Activation('relu')(bn)
    bn = layers.Dropout(0.2)(bn)

    # ── Decoder Level 3 ───────────────────────────────────────────────────
    u3 = layers.UpSampling2D(2)(bn)
    u3 = layers.Concatenate()([u3, c3])
    d3 = layers.Conv2D(128, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(u3)
    d3 = layers.BatchNormalization()(d3)
    d3 = layers.Activation('relu')(d3)
    d3 = layers.Conv2D(128, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(d3)
    d3 = layers.BatchNormalization()(d3)
    d3 = layers.Activation('relu')(d3)
    d3 = layers.Dropout(0.2)(d3)

    # ── Decoder Level 2 ───────────────────────────────────────────────────
    u2 = layers.UpSampling2D(2)(d3)
    u2 = layers.Concatenate()([u2, c2])
    d2 = layers.Conv2D(64, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(u2)
    d2 = layers.BatchNormalization()(d2)
    d2 = layers.Activation('relu')(d2)
    d2 = layers.Conv2D(64, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(d2)
    d2 = layers.BatchNormalization()(d2)
    d2 = layers.Activation('relu')(d2)

    # ── Decoder Level 1 ───────────────────────────────────────────────────
    u1 = layers.UpSampling2D(2)(d2)
    u1 = layers.Concatenate()([u1, c1])
    d1 = layers.Conv2D(32, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(u1)
    d1 = layers.BatchNormalization()(d1)
    d1 = layers.Activation('relu')(d1)
    d1 = layers.Conv2D(32, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(d1)
    d1 = layers.BatchNormalization()(d1)
    d1 = layers.Activation('relu')(d1)

    # ── Output — float32 for mixed-precision stability ────────────────────
    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32')(d1)

    model = Model(inputs, outputs, name='Document_UNet')
    return model

try:
    import gc
    tf.keras.backend.clear_session()
    gc.collect()
    model = build_unet(input_shape=(PATCH_SIZE, PATCH_SIZE, IMG_CHANNELS))
    model.summary()
    print(f'\n✅ Total parameters: {model.count_params():,}')
except Exception as e:
    print(f'❌ Model build error: {e}')
    raise

In [ ]:
def document_loss(y_true, y_pred):
    """Text-optimised composite loss: 0.5×L1 + 0.3×SSIM + 0.2×Edge."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    l1       = tf.reduce_mean(tf.abs(y_true - y_pred))
    ssim_val = 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

    true_edges = tf.image.sobel_edges(y_true)
    pred_edges = tf.image.sobel_edges(y_pred)
    edge_loss  = tf.reduce_mean(tf.abs(true_edges - pred_edges))

    return 0.5 * l1 + 0.3 * ssim_val + 0.2 * edge_loss

def psnr_metric_fn(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val=1.0))

def ssim_metric_fn(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

try:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=document_loss,
        metrics=[psnr_metric_fn, ssim_metric_fn]
    )
    print('✅ Model compiled')
except Exception as e:
    print(f'❌ Compilation error: {e}')
    raise

In [ ]:
import time

checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_model.keras')

# Check for existing checkpoint
initial_epoch = 0
try:
    if os.path.exists(checkpoint_path):
        print('🔄 Found existing checkpoint! Loading and resuming...')
        model = tf.keras.models.load_model(checkpoint_path, custom_objects={
            'document_loss':   document_loss,
            'psnr_metric_fn':  psnr_metric_fn,
            'ssim_metric_fn':  ssim_metric_fn
        })
        # Determine last epoch from training log
        log_path = os.path.join(CHECKPOINT_DIR, 'training_log.csv')
        if os.path.exists(log_path):
            import pandas as pd
            log = pd.read_csv(log_path)
            initial_epoch = len(log)
            print(f'  Resuming from epoch {initial_epoch}')
    else:
        print('🆕 Starting fresh training')
except Exception as e:
    print(f'⚠️  Could not load checkpoint ({e}) — starting fresh')
    initial_epoch = 0

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=30, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        os.path.join(CHECKPOINT_DIR, 'training_log.csv'), append=True
    )
]

remaining = EPOCHS - initial_epoch
steps_per_epoch = len(train_input_patches) // BATCH_SIZE
print(f'🏋️  Training for {remaining} epochs (batch size {BATCH_SIZE})')
print(f'   ~{steps_per_epoch} steps/epoch, ~10-15 sec/epoch on T4 GPU')
print(f'   Estimated total: ~{remaining * 15 // 60}-{remaining * 20 // 60} min')

try:
    start = time.time()
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        initial_epoch=initial_epoch,
        callbacks=callbacks
    )
    elapsed = time.time() - start
    print(f'\n✅ Training complete in {elapsed / 60:.1f} min')
    print(f'💾 Best model saved to: {checkpoint_path}')
except Exception as e:
    print(f'❌ Training error: {e}')
    raise

In [ ]:
try:
    h = history.history
    epochs_ran = range(initial_epoch + 1, initial_epoch + len(h['loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Training History', fontsize=14)

    axes[0].plot(epochs_ran, h['loss'],           label='Train Loss')
    axes[0].plot(epochs_ran, h['val_loss'],       label='Val Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

    psnr_key = [k for k in h if 'psnr' in k.lower() and 'val' not in k][0]
    val_psnr_key = 'val_' + psnr_key
    axes[1].plot(epochs_ran, h[psnr_key],     label='Train PSNR')
    axes[1].plot(epochs_ran, h[val_psnr_key], label='Val PSNR')
    axes[1].set_title('PSNR (dB)'); axes[1].legend(); axes[1].set_xlabel('Epoch')

    ssim_key = [k for k in h if 'ssim' in k.lower() and 'val' not in k][0]
    val_ssim_key = 'val_' + ssim_key
    axes[2].plot(epochs_ran, h[ssim_key],     label='Train SSIM')
    axes[2].plot(epochs_ran, h[val_ssim_key], label='Val SSIM')
    axes[2].set_title('SSIM'); axes[2].legend(); axes[2].set_xlabel('Epoch')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'training_history.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print('✅ Training history saved')

except Exception as e:
    print(f'⚠️  Could not plot history: {e}')

In [ ]:
def enhance_full_image(image_path, model, patch_size=128):
    """
    Enhance a full-size image using overlapping patch-based inference:
    1. Split into overlapping patches (50% overlap for smooth stitching)
    2. Run each through the model
    3. Average overlapping regions
    4. Return the full enhanced image (same size as input)
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f'Could not read: {image_path}')

    h, w = img.shape
    img_normalized = img.astype(np.float32) / 255.0

    # Pad to be divisible by patch_size
    pad_h = (patch_size - h % patch_size) % patch_size
    pad_w = (patch_size - w % patch_size) % patch_size
    img_padded = np.pad(img_normalized, ((0, pad_h), (0, pad_w)), mode='reflect')

    h_pad, w_pad = img_padded.shape
    output = np.zeros_like(img_padded)
    count  = np.zeros_like(img_padded)

    stride = patch_size // 2  # 50% overlap for smooth blending

    patches   = []
    positions = []
    for y in range(0, h_pad - patch_size + 1, stride):
        for x in range(0, w_pad - patch_size + 1, stride):
            patches.append(img_padded[y:y + patch_size, x:x + patch_size])
            positions.append((y, x))

    # Batch predict
    patches_array   = np.array(patches, dtype=np.float32)[..., np.newaxis]
    enhanced_patches = model.predict(patches_array, batch_size=32, verbose=0)

    # Stitch back with averaging
    for (y, x), ep in zip(positions, enhanced_patches):
        output[y:y + patch_size, x:x + patch_size] += ep[:, :, 0]
        count[y:y + patch_size,  x:x + patch_size] += 1.0

    output = output / np.maximum(count, 1.0)
    output = output[:h, :w]  # Remove padding
    output = np.clip(output * 255, 0, 255).astype(np.uint8)
    return output

print('✅ enhance_full_image() defined')

In [ ]:
try:
    # Load best model
    best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.keras')
    if os.path.exists(best_model_path):
        best_model = tf.keras.models.load_model(best_model_path, custom_objects={
            'document_loss':   document_loss,
            'psnr_metric_fn':  psnr_metric_fn,
            'ssim_metric_fn':  ssim_metric_fn
        })
        print(f'✅ Loaded best model from {best_model_path}')
    else:
        best_model = model
        print('ℹ️  Using last model (no checkpoint found)')

    n_show = min(10, len(val_files))
    dl_psnr_list = []
    dl_ssim_list = []

    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
    fig.suptitle('Validation Results: Input | Enhanced (U-Net) | Target', fontsize=13)

    for i, fname in enumerate(val_files[:n_show]):
        inp_path = os.path.join(VAL_INPUT_DIR,  fname)
        tgt_path = os.path.join(VAL_TARGET_DIR, fname)

        enhanced = enhance_full_image(inp_path, best_model, patch_size=PATCH_SIZE)
        tgt      = cv2.imread(tgt_path, cv2.IMREAD_GRAYSCALE)
        inp      = cv2.imread(inp_path, cv2.IMREAD_GRAYSCALE)

        # Resize target to match enhanced size for fair metric comparison
        tgt_r = cv2.resize(tgt, (enhanced.shape[1], enhanced.shape[0]))

        p = compute_psnr(enhanced, tgt_r)
        s = compute_ssim(enhanced, tgt_r)
        dl_psnr_list.append(p)
        dl_ssim_list.append(s)

        axes[i, 0].imshow(inp,      cmap='gray'); axes[i, 0].set_title('Input');       axes[i, 0].axis('off')
        axes[i, 1].imshow(enhanced, cmap='gray'); axes[i, 1].set_title(f'Enhanced\nPSNR={p:.1f} SSIM={s:.3f}'); axes[i, 1].axis('off')
        axes[i, 2].imshow(tgt,      cmap='gray'); axes[i, 2].set_title('Target');      axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'val_results.png'), dpi=100, bbox_inches='tight')
    plt.show()

    print(f'\n✅ Deep Learning Average Results:')
    print(f'   PSNR: {np.mean(dl_psnr_list):.2f} dB')
    print(f'   SSIM: {np.mean(dl_ssim_list):.4f}')

except Exception as e:
    print(f'❌ Inference error: {e}')
    raise

In [ ]:
try:
    all_methods  = list(classical_results.keys()) + ['U-Net']
    avg_psnr_all = [np.mean(classical_results[m]['psnr']) for m in classical_results]
    avg_ssim_all = [np.mean(classical_results[m]['ssim']) for m in classical_results]

    if dl_psnr_list:
        avg_psnr_all.append(np.mean(dl_psnr_list))
        avg_ssim_all.append(np.mean(dl_ssim_list))

    x = np.arange(len(all_methods))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('Method Comparison', fontsize=14)

    bars0 = axes[0].bar(x, avg_psnr_all, color=['steelblue'] * len(classical_results) + ['tomato'])
    axes[0].set_xticks(x); axes[0].set_xticklabels(all_methods, rotation=15)
    axes[0].set_ylabel('PSNR (dB)'); axes[0].set_title('Average PSNR')
    for bar, v in zip(bars0, avg_psnr_all):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                     f'{v:.1f}', ha='center', va='bottom', fontsize=9)

    bars1 = axes[1].bar(x, avg_ssim_all, color=['steelblue'] * len(classical_results) + ['tomato'])
    axes[1].set_xticks(x); axes[1].set_xticklabels(all_methods, rotation=15)
    axes[1].set_ylabel('SSIM'); axes[1].set_title('Average SSIM')
    for bar, v in zip(bars1, avg_ssim_all):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                     f'{v:.3f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'method_comparison.png'), dpi=100, bbox_inches='tight')
    plt.show()

    best_idx  = int(np.argmax(avg_psnr_all))
    print(f'🏆 Best method by PSNR: {all_methods[best_idx]} ({avg_psnr_all[best_idx]:.2f} dB)')

except Exception as e:
    print(f'⚠️  Comparison chart error: {e}')

In [ ]:
# ===== ENHANCE YOUR OWN IMAGE =====
# Change the path below to any medical report image on your Drive
test_image_path = '/content/drive/MyDrive/your_medical_report.png'

try:
    enhanced = enhance_full_image(test_image_path, best_model, patch_size=PATCH_SIZE)

    save_path = os.path.join(RESULTS_DIR, 'my_enhanced.png')
    cv2.imwrite(save_path, enhanced)
    print(f'✅ Enhanced image saved to: {save_path}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    orig = cv2.imread(test_image_path, cv2.IMREAD_GRAYSCALE)
    axes[0].imshow(orig,     cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(enhanced, cmap='gray'); axes[1].set_title('Enhanced'); axes[1].axis('off')
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f'ℹ️  No test image found at {test_image_path}')
    print('    Update test_image_path above to point to your image.')
except Exception as e:
    print(f'❌ Enhancement error: {e}')

In [ ]:
import csv

try:
    # Save metrics summary to CSV
    csv_path = os.path.join(RESULTS_DIR, 'metrics_summary.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Method', 'Avg_PSNR', 'Avg_SSIM'])
        for name in classical_results:
            writer.writerow([
                name,
                f'{np.mean(classical_results[name]["psnr"]):.4f}',
                f'{np.mean(classical_results[name]["ssim"]):.4f}'
            ])
        if dl_psnr_list:
            writer.writerow(['U-Net',
                             f'{np.mean(dl_psnr_list):.4f}',
                             f'{np.mean(dl_ssim_list):.4f}'])
    print(f'✅ Metrics saved to: {csv_path}')

except Exception as e:
    print(f'⚠️  Metrics save error: {e}')

# Free GPU memory
try:
    tf.keras.backend.clear_session()
    gc.collect()
    print('✅ Memory cleared')
except Exception:
    pass

print('\n🎉 All done!')
print(f'   Results folder: {RESULTS_DIR}')
print(f'   Checkpoint:     {CHECKPOINT_DIR}')